# A Modern Training Loop: Datasets and DataLoaders, CIFAR10, Mixed Precision, torch.compile

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/03-Modern-Training-Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_iterable_shard.py', 'q_amp_benchmark.py', 'q_tta.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
import os, sys
# DataLoader workers: Colab (Linux) has 2 CPU cores. On Windows/macOS, Dataset classes defined in a
# notebook cannot be used by workers (see "How are workers started?" in 03), so we use 0 there.
NUM_WORKERS = 2 if sys.platform == "linux" else 0
if sys.platform == "linux":
    torch.multiprocessing.set_start_method("fork", force=True)   # the default changed to "forkserver" in Python 3.14
torch.backends.cudnn.benchmark = True   # with fixed input sizes, let cuDNN pick the fastest algorithm

## CIFAR10

Ten classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck), 32x32 color images.

In [ ]:
# Download from the Hugging Face Hub (much faster than the original CIFAR site)
from datasets import load_dataset
from types import SimpleNamespace
def load_cifar10(split):
    ds = load_dataset("uoft-cs/cifar10", split=split)
    return SimpleNamespace(data=np.stack([np.array(im) for im in ds["img"]]), targets=list(ds["label"]))
train_raw = load_cifar10("train")
test_raw = load_cifar10("test")
name_array = np.array(["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"])
train_raw.data.shape, len(train_raw.targets)

In [ ]:
def showX(X):
    """X: (N, 32, 32, 3) uint8"""
    display(Image.fromarray(np.concatenate(list(X), axis=1)).resize((64 * len(X), 64), Image.NEAREST))
showX(train_raw.data[:12])
print(name_array[train_raw.targets[:12]])

## Dataset and DataLoader

Training reads the data in **mini-batches**, usually in a **shuffled** order that changes every epoch. PyTorch splits this job in two:

* a **`Dataset`** knows how to produce *one sample*;
* a **`DataLoader`** decides the order (shuffle), groups samples into batches, and can prepare batches in parallel in background processes (**workers**).

There are two kinds of `Dataset`, and they handle shuffling, batching and workers very differently. Let's look at both with a tiny dataset of the numbers 0–9, so we can see exactly what comes out.

### 1. Map-style dataset: `__getitem__` + `__len__`

You can ask for sample `i` directly (random access), and you know how many samples there are.

In [ ]:
from torch.utils.data import Dataset, IterableDataset, DataLoader, get_worker_info

def worker_id():
    info = get_worker_info()          # None in the main process
    return -1 if info is None else info.id

class Numbers(Dataset):
    def __init__(self, n=10):
        self.n = n
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        return i, worker_id()          # the sample, and which process produced it

loader = DataLoader(Numbers(), batch_size=4, shuffle=True)
for epoch in range(2):
    print(f"epoch {epoch}:", [batch[0].tolist() for batch in loader])

How it works:

* **Shuffle** is done by a **sampler** in the main process. With `shuffle=True` the DataLoader uses a `RandomSampler`, which makes a new random permutation of the indices `0..len-1` **every epoch** (compare the two epochs above). With `shuffle=False` the indices are simply `0, 1, 2, ...`.
* **Batching** is done by a **batch sampler**, which cuts that index sequence into groups of `batch_size`. The last batch is smaller (2 samples here) unless you pass `drop_last=True`.
* For each group of indices the DataLoader calls `dataset[i]` and then **`collate_fn`** stacks the samples into tensors (numbers become a 1-D tensor; images of shape `(3, 32, 32)` become `(batch, 3, 32, 32)`).

We can look at the indices directly:

In [ ]:
print("index batches:", list(loader.batch_sampler))
print("drop_last=True:", [b[0].tolist() for b in DataLoader(Numbers(), batch_size=4, shuffle=True, drop_last=True)])

With **workers**, the main process still runs the sampler and hands out *whole batches of indices* to the workers, round-robin. Every index is used exactly once, so nothing is duplicated, and the batches come back in the sampler's order. The second number shows which worker produced each sample (`-1` = the main process):

In [ ]:
loader = DataLoader(Numbers(), batch_size=4, shuffle=True, num_workers=NUM_WORKERS)
for numbers, workers in loader:
    print("numbers", numbers.tolist(), " produced by worker", workers.tolist())

### 2. Iterable-style dataset: `__iter__`

Some data cannot be indexed: a stream from the network, lines of a huge file, data generated on the fly, or a dataset bigger than memory that must be read sequentially (for example `load_dataset(..., streaming=True)` in 🤗 datasets). An `IterableDataset` only implements `__iter__`, which yields samples one after another. There is no `len()` and no `dataset[i]`.

In [ ]:
class NumberStream(IterableDataset):
    def __init__(self, n=10):
        self.n = n
    def __iter__(self):
        for i in range(self.n):
            yield i, worker_id()

loader = DataLoader(NumberStream(), batch_size=4)
print([batch[0].tolist() for batch in loader])

**Batching** still works: the DataLoader takes samples from the iterator and groups every `batch_size` of them (the last batch is smaller here too).

**Shuffling** does not: without indices there is no sampler to permute, so `shuffle=True` is an error:

In [ ]:
try:
    DataLoader(NumberStream(), batch_size=4, shuffle=True)
except ValueError as e:
    print("ValueError:", e)

The usual fix is a **shuffle buffer** inside the dataset: keep `buffer_size` samples in memory and output a random one of them each time. It only shuffles *locally*: a sample can move at most about `buffer_size` positions, so the buffer must be large compared with how "sorted" the stream is. (This is what `IterableDataset.shuffle(buffer_size=...)` does in 🤗 datasets.)

In [ ]:
import random

def shuffle_buffer(iterator, buffer_size):
    buf = []
    for item in iterator:
        buf.append(item)
        if len(buf) >= buffer_size:
            i = random.randrange(len(buf))
            buf[i], buf[-1] = buf[-1], buf[i]
            yield buf.pop()
    random.shuffle(buf)
    yield from buf

class ShuffledStream(IterableDataset):
    def __init__(self, n=10, buffer_size=4):
        self.n, self.buffer_size = n, buffer_size
    def __iter__(self):
        return shuffle_buffer(((i, worker_id()) for i in range(self.n)), self.buffer_size)

print("buffer 3:  ", list(next(iter(DataLoader(ShuffledStream(30, 3), batch_size=30)))[0].tolist()))
print("buffer 30: ", list(next(iter(DataLoader(ShuffledStream(30, 30), batch_size=30)))[0].tolist()))

**Workers** are the big trap of iterable-style datasets. Each worker receives its own *copy* of the dataset object and runs its own `__iter__` from the beginning, so with 2 workers **every sample appears twice**:

In [ ]:
loader = DataLoader(NumberStream(), batch_size=4, num_workers=NUM_WORKERS)
for numbers, workers in loader:
    print("numbers", numbers.tolist(), " produced by worker", workers.tolist())

(If you run this locally on Windows with `NUM_WORKERS = 0` you won't see the duplicates; on Colab you will.)

Each worker must therefore yield only **its own share** (a *shard*) of the data, using `get_worker_info()`, which tells the worker its `id` and the total `num_workers`.

## Q
Write `ShardedStream`, an `IterableDataset` that yields the numbers `0..n-1` exactly once in total even with several workers (for example, worker `k` of `W` takes the numbers `k, k+W, k+2W, ...`). Check that with `num_workers=2` every number appears exactly once. In what order do the batches come out?

In [ ]:
# %load q_iterable_shard.py
%run -i q_iterable_shard.py

### Summary: the two kinds of dataset

| | Map-style (`__getitem__`, `__len__`) | Iterable-style (`__iter__`) |
|---|---|---|
| Typical data | in memory, or files you can index | streams, huge files, generated data |
| Shuffle | `shuffle=True`: the sampler permutes all indices every epoch (exact) | not allowed in `DataLoader`; use a shuffle buffer inside (approximate) |
| Batch | batch sampler groups indices, then `collate_fn` stacks | DataLoader groups consecutive samples, then `collate_fn` stacks |
| `len(loader)` | known | unknown (unless you define it) |
| Workers | the main process hands out index batches, no duplicates | each worker runs its own `__iter__`, so you must shard with `get_worker_info()` |

### What do workers change?

* `num_workers=0`: samples are loaded **in the main process**, one batch at a time, *between* training steps, so the GPU waits while the CPU prepares the next batch.
* `num_workers=k`: `k` background processes prepare batches **in parallel and ahead of time** (each keeps `prefetch_factor=2` batches ready by default), so loading overlaps with GPU computation. More workers than CPU cores does not help; Colab has only 2.
* Every worker holds its **own copy of the dataset object**: a dataset that keeps everything in memory is copied once per worker (on Linux, fork shares memory until it is written to).
* Starting the workers costs time at the start of **every epoch**; `persistent_workers=True` keeps them alive between epochs.
* Random augmentation in workers: PyTorch seeds `torch`, Python's `random` and `np.random` differently in each worker, so `transforms.v2` and code using those global generators get different random numbers in each worker. The trap is a generator stored **inside the dataset object** (e.g. `self.rng = np.random.default_rng(0)` in `__init__`): every worker gets a copy of the same state and produces the *same* "random" numbers. Re-seed it in `worker_init_fn` using `get_worker_info().seed`.
* `pin_memory=True` puts each batch in page-locked memory, which lets `.to("cuda", non_blocking=True)` copy it to the GPU in the background.
* How many workers? Start with the number of CPU cores (`os.cpu_count()`; 2 on Colab) and **measure**: more workers than cores only adds memory use and contention.

### How are workers started? fork, spawn and forkserver

Each worker is a separate operating-system process, so the dataset object and your code have to get into it somehow. Python's `multiprocessing` has three **start methods**:

* **fork** — the child process is a copy of the parent (copy-on-write memory): every object, class and function that exists in the notebook is already there, nothing needs to be serialized, and starting is fast. Caveats: the copy includes the parent's threads and locks in whatever state they happen to be in, which can deadlock in multi-threaded programs; and a forked child cannot use CUDA if the parent has already initialized it, so **workers should only do CPU work** (load, decode, augment) and return CPU tensors.
* **spawn** — the child starts a brand-new Python interpreter, imports your main module, and receives the dataset, `collate_fn` and `worker_init_fn` **pickled** from the parent. Everything must therefore be picklable and importable: classes defined in a notebook cell or in `if __name__ == "__main__":` code, and lambdas, do not work. Starting is slow (each worker re-imports PyTorch), which makes `persistent_workers=True` more important.
* **forkserver** — a small "server" process is started once (with spawn); each worker is then forked from that clean server instead of from your busy main process. It avoids most of fork's thread problems, but data is still passed by pickling, with the same restrictions as spawn.

| | Default for | Dataset/classes passed by | Notebook-defined classes |
|---|---|---|---|
| fork | Linux with Python ≤ 3.13 (Colab today) | copied memory | ✅ |
| spawn | Windows, macOS | pickle | ❌ (put them in a `.py` file) |
| forkserver | Linux with Python ≥ 3.14 | pickle | ❌ |

You can choose per loader with `DataLoader(..., multiprocessing_context="fork" | "spawn" | "forkserver")`, or globally with `torch.multiprocessing.set_start_method(...)`. This notebook sets `"fork"` on Linux in its first cell so the examples keep working when Colab moves to Python 3.14, and uses `NUM_WORKERS = 0` on Windows and macOS.

Further reading:
* PyTorch: [`torch.utils.data` overview](https://docs.pytorch.org/docs/stable/data.html), especially the sections *Multi-process data loading*, *Platform-specific behaviors*, *Randomness in multi-process data loading* and *Memory Pinning*
* PyTorch: [`IterableDataset` and sharding with `get_worker_info()`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.IterableDataset)
* Python: [`multiprocessing` — Contexts and start methods](https://docs.python.org/3/library/multiprocessing.html#contexts-and-start-methods)
* PyTorch tutorial: [Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)
* 🤗 datasets: [Streaming (iterable datasets, shuffle buffers, sharding)](https://huggingface.co/docs/datasets/stream)

### A map-style dataset for CIFAR10

`torchvision.transforms.v2` is the new image transform API. The random crop and flip are **data augmentation**: every epoch sees slightly different versions of each image.

In [ ]:
import torchvision.transforms.v2 as T
mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.2470, 0.2435, 0.2616])
train_tf = T.Compose([
    T.ToImage(),                                   # numpy HWC uint8 -> tensor CHW uint8
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToDtype(torch.float32, scale=True),          # uint8 [0, 255] -> float [0, 1]
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(mean, std)])

class CIFAR10Dataset(Dataset):
    def __init__(self, data, targets, transform=None):
        self.data = data            # (N, 32, 32, 3) uint8 numpy array
        self.targets = targets      # list of int
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        x = self.data[i]
        if self.transform is not None:
            x = self.transform(x)
        return x, self.targets[i]

train_ds = CIFAR10Dataset(train_raw.data, train_raw.targets, train_tf)
test_ds = CIFAR10Dataset(test_raw.data, test_raw.targets, test_tf)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=NUM_WORKERS > 0)
xb, yb = next(iter(train_loader))
print(xb.shape, xb.dtype, yb.shape, "batches per epoch:", len(train_loader))

How fast can this loader produce batches (no training, just loading)? Compare with and without workers:

In [ ]:
def loader_speed(loader, max_batches=40):
    t0, n = time.perf_counter(), 0
    for i, (xb, yb) in enumerate(loader):
        n += len(xb)
        if i + 1 == max_batches:
            break
    return n / (time.perf_counter() - t0)

for w in sorted({0, NUM_WORKERS}):
    dl = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=w)
    print(f"num_workers={w}: {loader_speed(dl):,.0f} img/s")

### Keeping the data on the GPU

CIFAR10 is small (50000x3x32x32 floats is about 600MB), so there is another option: put the whole dataset on the GPU once, and do the shuffling, batching and augmentation there with tensor operations on whole batches.
This is the iterator style again, but much simpler: a Python generator that shuffles **indices** with `torch.randperm` (exact shuffle), slices them into batches, and augments a whole batch at once. No workers are needed, because there is no per-sample CPU work left.

`GPULoader` wraps it with the same interface as a `DataLoader` (iterate over it to get batches; `len()` is the number of batches), so the training code below can use either.

In [ ]:
def to_tensor(raw):
    X = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)   # NHWC -> NCHW
    X = (X - torch.tensor(mean).view(1, 3, 1, 1).float()) / torch.tensor(std).view(1, 3, 1, 1).float()
    return X.to(device), torch.tensor(raw.targets).to(device)
train_X, train_y = to_tensor(train_raw)
test_X, test_y = to_tensor(test_raw)

def gpu_augment(X, pad=4):
    """random crop (reflect padding) + random horizontal flip, processing the whole batch at once on the GPU"""
    N, C, H, W = X.shape
    Xp = F.pad(X, (pad,) * 4, mode="reflect")
    dx, dy = torch.randint(0, 2 * pad + 1, (2, N), device=X.device)
    rows = (torch.arange(H, device=X.device)[None, :] + dy[:, None])        # N x H
    cols = (torch.arange(W, device=X.device)[None, :] + dx[:, None])        # N x W
    X = Xp[torch.arange(N, device=X.device)[:, None, None, None],
           torch.arange(C, device=X.device)[None, :, None, None],
           rows[:, None, :, None], cols[:, None, None, :]]
    flip = torch.rand(N, device=X.device) < 0.5
    return torch.where(flip[:, None, None, None], X.flip(3), X)

def batches(X, y, batch_size, shuffle=False, augment=False):
    idx = torch.randperm(len(X), device=X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), batch_size):
        j = idx[i:i + batch_size]
        xb = X[j]
        yield (gpu_augment(xb) if augment else xb), y[j]

class GPULoader:
    def __init__(self, X, y, batch_size, shuffle=False, augment=False):
        self.X, self.y, self.batch_size, self.shuffle, self.augment = X, y, batch_size, shuffle, augment
    def __len__(self):
        return (len(self.X) + self.batch_size - 1) // self.batch_size
    def __iter__(self):
        return batches(self.X, self.y, self.batch_size, self.shuffle, self.augment)

train_gpu_loader = GPULoader(train_X, train_y, 512, shuffle=True, augment=True)
print(f"GPULoader: {loader_speed(train_gpu_loader):,.0f} img/s")

In [ ]:
xb, yb = next(batches(train_X, train_y, 12, augment=True))
img = (xb.permute(0, 2, 3, 1).cpu().numpy() * std + mean).clip(0, 1)
showX((img * 255).astype(np.uint8))

## Model: a small ResNet

* BatchNorm stabilizes training
* Residual connections make deep networks trainable

In [ ]:
def conv_bn(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.GELU(),
    )

class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(conv_bn(c, c), conv_bn(c, c))
    def forward(self, x):
        return x + self.block(x)

def resnet9(num_classes=10, w=64):
    return nn.Sequential(
        conv_bn(3, w),
        conv_bn(w, 2 * w), nn.MaxPool2d(2), Residual(2 * w),      # 16x16
        conv_bn(2 * w, 4 * w), nn.MaxPool2d(2),                   # 8x8
        conv_bn(4 * w, 8 * w), nn.MaxPool2d(2), Residual(8 * w),  # 4x4
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),
        nn.Linear(8 * w, num_classes),
    )
model = resnet9()
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Mixed precision training (Automatic Mixed Precision, AMP)

* `torch.autocast`: automatically switches suitable ops (matmul, conv) to fp16 and keeps the rest (e.g. softmax, loss) in fp32
* `GradScaler`: fp16 has a small range, so very small gradients become 0 (underflow). It scales the loss up before backward and scales the gradients back down before the update
* On A100/H100/RTX 30-series or newer cards you can use `dtype=torch.bfloat16` instead and skip the GradScaler. **The T4 only supports fp16**

In [ ]:
@torch.no_grad()
def evaluate(model, X=None, y=None, amp=True):
    X = test_X if X is None else X
    y = test_y if y is None else y
    model.eval()
    correct = 0
    for xb, yb in batches(X, y, 1000):
        with torch.autocast(device, dtype=torch.float16, enabled=amp):
            correct += (model(xb.contiguous(memory_format=torch.channels_last)).argmax(1) == yb).sum().item()
    return correct / len(X)

def fit(model, train_loader, epochs=10, lr=2e-3, amp=True, compile=False, label_smoothing=0.1):
    """train_loader: anything we can iterate over to get (images, labels) batches, with len() = batches per epoch"""
    model = model.to(device, memory_format=torch.channels_last)   # channels_last is faster for Tensor Core convolutions
    train_model = torch.compile(model) if compile else model
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4, fused=(device == "cuda"))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader))
    scaler = torch.amp.GradScaler(device, enabled=amp)
    for epoch in range(epochs):
        model.train()
        t0, total_loss, n_images = time.perf_counter(), 0.0, 0
        for xb, yb in train_loader:
            # a no-op for GPULoader; for a DataLoader this copies the batch from (pinned) CPU memory to the GPU
            xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
            yb = yb.to(device, non_blocking=True)
            with torch.autocast(device, dtype=torch.float16, enabled=amp):
                loss = F.cross_entropy(train_model(xb), yb, label_smoothing=label_smoothing)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            n_images += len(xb)
        if device == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        print(f"epoch {epoch:2d}  loss={total_loss/len(train_loader):.3f}  test acc={evaluate(model, amp=amp):.4f}"
              f"  {dt:.1f}s ({n_images/dt:.0f} img/s)")
    return model

### Training with the DataLoader

First, 2 epochs with the ordinary `DataLoader` (augmentation done by the CPU workers):

In [ ]:
model = fit(resnet9(), train_loader, epochs=2)

### Training with the data on the GPU

Now the same model and training code with `GPULoader`, for 10 epochs. Compare the img/s with the DataLoader run above: on Colab, with only 2 CPU cores, the CPU workers cannot prepare images as fast as the T4 can train on them, so the GPU spends time waiting.
For a dataset that does not fit in GPU memory (like the 224x224 photos in the next notebook) the `DataLoader` with workers is the standard approach.

In [ ]:
model = fit(resnet9(), train_gpu_loader, epochs=10)

10 epochs should reach about 90% accuracy.

In [ ]:
# Look at the predictions
with torch.no_grad(), torch.autocast(device, dtype=torch.float16):
    pred = model(test_X[:12]).argmax(1).cpu()
showX(test_raw.data[:12])
print("pred:", name_array[pred])
print("label:", name_array[test_raw.targets[:12]])

## Q
Compare the following settings, running only 2 epochs each, and see how many images per second (img/s) they process:
* `amp=False` (pure fp32)
* `amp=True`
* `amp=True, compile=True` (the first epoch includes compile time, so look at the second epoch)

`torch.compile` compiles the model into optimized kernels (using Triton), so the first run is slower.

Note: **always measure** optimizations. On the T4, AMP is roughly 3-4x faster, but `torch.compile` may actually slow down this small model
(cuDNN convolutions are already fast, and the T4 doesn't have enough SMs for some autotuning modes). Compile usually helps more with large models or models with many small elementwise ops.

In [ ]:
# %load q_amp_benchmark.py
%run -i q_amp_benchmark.py

## Saving a checkpoint

To resume training later, save the optimizer, scheduler, and scaler states in addition to the model parameters.
Here we only show the basics: saving the model parameters.

In [ ]:
torch.save(model.state_dict(), "resnet9_cifar10.pt")
model2 = resnet9().to(device, memory_format=torch.channels_last)
model2.load_state_dict(torch.load("resnet9_cifar10.pt"))
evaluate(model2)

## Q: Test Time Augmentation (TTA)
At prediction time, feed both the original image and its horizontal flip into the model and average the two outputs (logits or probabilities). Does accuracy improve?

In [ ]:
# %load q_tta.py
%run -i q_tta.py